# UrbanRisk AI — M4: Urban Risk Engine + Anomaly Detection
**Milestone 4 of 9** — combines M1 (AQI), M2 (flood), M3 (traffic) model outputs into one 0-100 Urban Risk Score, and adds an Isolation Forest anomaly layer.

### Scope of this notebook — and what it deliberately does NOT do
This milestone's job is the **combination logic**, not live data ingestion (that's M6). So this notebook:
- Loads the actual trained M1/M2/M3 models and their real metadata
- Builds the score-combination and anomaly-detection functions
- Demonstrates them on a handful of **clearly-labeled illustrative scenarios** built from real static
  per-zone features (elevation, historical inundation, road density, hotspot distance) + hypothetical
  weather/traffic conditions — not live-fetched data
- Saves the engine config so M6 can call it with real feature rows later

### Four honesty notes worth reading before the code

**1. Flood model selection.** M2's saved metrics show 0.5-threshold Precision/Recall/F1 are close to
meaningless here (e.g. `xgboost_12h` has Precision=Recall=F1=0.0) — a direct consequence of how rare the
proxy-positive hours are, not a bug. ROC-AUC and PR-AUC still rank the models meaningfully. So this engine
(a) uses **raw predicted probabilities** as a continuous risk contribution, never a 0.5 threshold, and
(b) picks whichever of {LogReg, RF, XGBoost} actually has the best test ROC-AUC **per horizon**, read
dynamically from `flood_model_metadata.json` — a real comparison, not an assumed "XGBoost is always best."

**2. AQI rescaling.** The AQI model predicts a standardized 0–500 composite index value. This engine
rescales it to a 0–100 risk contribution via `min(100, AQI/5)` — a direct linear rescale of the existing
index, not a new methodology.

**3. `zone_id` encoding assumption.** The flood RandomForest confirms one-hot `zone_id_CHN_*` columns (no
ambiguity there). The AQI and flood **XGBoost** models were (per the M2/M3 pattern) trained on
`zone_id.astype("category").cat.codes` — pandas' default categorical encoding, which assigns codes by
**alphabetically sorted unique zone_id strings**. This notebook reproduces that exact ordering. If AQI/flood
XGBoost predictions look wrong for a specific zone, this is the first thing to verify against your actual M1
training code.

**4. Combination weights are a documented judgment call, not an official standard.** Default here: AQI 0.35,
Flood 0.40, Traffic 0.25 — flood weighted highest because GCC's own historical inundation data (real, used as
static evidence in M2) reflects acute safety risk; AQI next for chronic health burden; traffic lowest since
it's rarely acutely dangerous. Change `RISK_WEIGHTS` below to whatever your project defends instead.


## 1. Load M1/M2/M3 artifacts
Upload all of these to this Colab session (or mount Drive and adjust paths): the AQI/flood/traffic `.joblib` models, `zones.csv`, `flood_zones_static_features.csv`, `model_metadata_multizone.json`, `flood_model_metadata.json`, plus M3's `traffic_zones_road_features.csv`, `traffic_zones_hotspot_distance.csv`, `traffic_model_metadata.json` and its three `.joblib` files.

In [ ]:
!pip install -q xgboost scikit-learn pandas numpy

import json
import joblib
import numpy as np
import pandas as pd
import re

pd.set_option("display.max_columns", 60)

zones_df = pd.read_csv("zones.csv")
flood_static_df = pd.read_csv("flood_zones_static_features.csv")

with open("model_metadata_multizone.json") as f:
    aqi_metadata = json.load(f)
with open("flood_model_metadata.json") as f:
    flood_metadata = json.load(f)
with open("traffic_model_metadata.json") as f:
    traffic_metadata = json.load(f)

# AQI models — regression, one per horizon (XGBoost was the chosen final model in M1)
aqi_models = {
    6:  joblib.load("aqi_xgboost_multizone_6h.joblib"),
    12: joblib.load("aqi_xgboost_multizone_12h.joblib"),
    24: joblib.load("aqi_xgboost_multizone_24h.joblib"),
}

# Flood models — classification, three algorithms per horizon (selection logic in Section 3)
flood_models = {
    6:  {"logreg": joblib.load("flood_logreg_6h.joblib"),
         "rf":     joblib.load("flood_rf_6h.joblib"),
         "xgboost": joblib.load("flood_xgboost_6h.joblib")},
    12: {"logreg": joblib.load("flood_logreg_12h.joblib"),
         "rf":     joblib.load("flood_rf_12h.joblib"),
         "xgboost": joblib.load("flood_xgboost_12h.joblib")},
}

# Traffic models — classification, three algorithms per horizon (same selection logic)
traffic_models = {
    1: {"logreg": joblib.load("congestion_logreg_1h.joblib"),
        "rf":     joblib.load("congestion_rf_1h.joblib"),
        "xgboost": joblib.load("congestion_xgboost_1h.joblib")},
    2: {"logreg": joblib.load("congestion_logreg_2h.joblib"),
        "rf":     joblib.load("congestion_rf_2h.joblib"),
        "xgboost": joblib.load("congestion_xgboost_2h.joblib")},
}

traffic_road_df = pd.read_csv("traffic_zones_road_features.csv")
traffic_hotspot_df = pd.read_csv("traffic_zones_hotspot_distance.csv")

# One master per-zone static table: real coordinates + real flood evidence + real road/hotspot evidence
zone_static_df = (
    zones_df
    .merge(flood_static_df[["zone_id", "elevation_m", "water_body_distance_km",
                             "historical_inundation_count", "historical_max_depth_in",
                             "historical_mean_depth_in"]], on="zone_id", how="left")
    .merge(traffic_road_df, on="zone_id", how="left")
    .merge(traffic_hotspot_df, on="zone_id", how="left")
)

assert set(zones_df["zone_id"]) == set(aqi_metadata["zone_ids"]), "Zone mismatch between zones.csv and AQI metadata"
print(f"{len(zone_static_df)} zones loaded with real static evidence merged in.")
zone_static_df.head()


## 2. Dynamic model selection — read the real comparison, don't assume
For flood and traffic, pick whichever algorithm actually had the best test ROC-AUC at each horizon, read straight from the saved metadata (case-insensitive, since M2's keys and M3's keys were saved with different casing).

In [ ]:
def _get_ci(d, key):
    \"\"\"Case-insensitive dict lookup — M2's metadata used 'ROC_AUC', M3's used 'roc_auc'.\"\"\"
    for k, v in d.items():
        if k.lower() == key.lower():
            return v
    raise KeyError(f"{key} not found in {list(d.keys())}")

def pick_best_model(metrics_dict, horizon_suffix):
    \"\"\"metrics_dict: {'logreg_6h': {...}, 'rf_6h': {...}, 'xgboost_6h': {...}}
    LogReg is excluded from the live pool: its saved .joblib does NOT include the fitted
    StandardScaler it was trained against, so feeding it raw feature values at inference time
    produces garbage probabilities (verified while building this notebook — one test row came
    back ~2e-220 instead of a real probability). Metadata's own ROC-AUC numbers for logreg are
    still valid — that's from M2/M3's own scaled test set — this is purely an inference-time gap.
    Fix: re-save the scaler (joblib.dump(scaler, ...)) in M2/M3 and re-enable logreg here.\"\"\"
    candidates = {
        k: v for k, v in metrics_dict.items()
        if k.endswith(f"_{horizon_suffix}") and not k.startswith("logreg")
    }
    best_key = max(candidates, key=lambda k: _get_ci(candidates[k], "roc_auc"))
    algo = best_key.rsplit("_", 1)[0]
    comparison = ", ".join(
        f"{k.rsplit('_', 1)[0]}={_get_ci(v, 'roc_auc'):.3f}" for k, v in candidates.items()
    )
    print(f"  {horizon_suffix}: {algo} wins on ROC-AUC among rf/xgboost ({comparison}) — logreg excluded, no saved scaler")
    return algo

print("Flood — best model per horizon:")
flood_best_algo = {h: pick_best_model(flood_metadata["test_metrics"], f"{h}h") for h in [6, 12]}

print("\nTraffic — best model per horizon:")
traffic_best_algo = {h: pick_best_model(traffic_metadata["test_metrics"], f"{h}h") for h in [1, 2]}


## 3. Feature-vector construction
A generic builder that reads each model's `feature_columns` list and fills every column from a small scenario dict + the real per-zone static table — so it works for AQI's 74-column schema and flood's 33-column schema without hand-writing either one.

In [ ]:
ZONE_ID_CODE_MAP = {zid: i for i, zid in enumerate(sorted(aqi_metadata["zone_ids"]))}  # see honesty note 3

def _parse_feature_name(name):
    m = re.match(r"^(.*)_lag(\d+)h$", name)
    if m: return m.group(1), "lag"
    m = re.match(r"^(.*)_rollmean(\d+)h$", name)
    if m: return m.group(1), "rollmean"
    m = re.match(r"^(.*)_rollstd(\d+)h$", name)
    if m: return m.group(1), "rollstd"
    m = re.match(r"^rainfall_past_(\d+)h$", name)
    if m: return "rainfall", ("sum", int(m.group(1)))
    m = re.match(r"^rainfall_(\d+)h$", name)
    if m: return "rainfall", ("sum", int(m.group(1)))
    return name, "raw"

def build_feature_row(feature_columns, scenario, zone_row):
    \"\"\"scenario: dict of base signal values (AQI, PM2_5, PM10, temperature, humidity, wind_speed,
    pressure, rainfall_rate_mm_per_h) + a 'calendar' sub-dict (hour, day_of_week, month, is_weekend).
    zone_row: one row of zone_static_df — the real static evidence for this zone.\"\"\"
    row = {}
    for col in feature_columns:
        if col == "zone_id":
            row[col] = ZONE_ID_CODE_MAP[zone_row["zone_id"]]
        elif col in ("latitude", "longitude"):
            row[col] = zone_row[col]
        elif col in scenario.get("calendar", {}):
            row[col] = scenario["calendar"][col]
        elif col in zone_row.index:
            row[col] = zone_row[col]
        else:
            base, kind = _parse_feature_name(col)
            if isinstance(kind, tuple) and kind[0] == "sum":
                row[col] = scenario.get("rainfall_rate_mm_per_h", 0.0) * kind[1]
            elif kind == "rollstd":
                row[col] = scenario.get(f"{base}_std", 0.0)
            elif kind in ("lag", "rollmean", "raw"):
                row[col] = scenario.get(base, scenario.get(col, 0.0))
            else:
                row[col] = 0.0
    return row

print("Feature builder ready.")


## 4. Risk-contribution functions per signal

In [ ]:
def aqi_to_risk(predicted_aqi):
    \"\"\"Linear rescale of the standardized 0-500 AQI index to a 0-100 risk contribution.\"\"\"
    return float(np.clip(predicted_aqi / 5.0, 0, 100))

def prob_to_risk(predicted_prob):
    \"\"\"Predicted probability (0-1) direct-scaled to a 0-100 risk contribution.\"\"\"
    return float(np.clip(predicted_prob * 100.0, 0, 100))

RISK_WEIGHTS = {"aqi": 0.35, "flood": 0.40, "traffic": 0.25}  # see honesty note 4 — adjust freely

RISK_CATEGORIES = [
    (20,  "Low"),
    (40,  "Moderate"),
    (60,  "Elevated"),
    (80,  "High"),
    (100, "Critical"),
]

def categorize_risk(score):
    for threshold, label in RISK_CATEGORIES:
        if score <= threshold:
            return label
    return "Critical"

def combine_urban_risk(aqi_risk, flood_risk, traffic_risk, weights=RISK_WEIGHTS):
    total = (weights["aqi"] * aqi_risk + weights["flood"] * flood_risk + weights["traffic"] * traffic_risk)
    total = float(np.clip(total, 0, 100))
    return total, categorize_risk(total)


## 5. Illustrative scenarios
Three hand-picked, clearly hypothetical Chennai weather/traffic conditions — **not live data** — used only to exercise the pipeline end to end. Real feature rows arrive from M6's live pipeline later, in this exact schema.

In [ ]:
SCENARIOS = {
    "Typical Dry Midday": {
        "AQI": 95, "PM2_5": 45, "PM10": 80, "temperature": 31, "humidity": 55,
        "wind_speed": 12, "pressure": 1008, "rainfall_rate_mm_per_h": 0.0,
        "calendar": {"hour": 13, "day_of_week": 2, "month": 3, "is_weekend": 0},
    },
    "Evening Rush, Moderate Rain": {
        "AQI": 130, "PM2_5": 65, "PM10": 110, "temperature": 27, "humidity": 85,
        "wind_speed": 18, "pressure": 1002, "rainfall_rate_mm_per_h": 8.0,
        "calendar": {"hour": 18, "day_of_week": 3, "month": 10, "is_weekend": 0},
    },
    "Northeast Monsoon Heavy Rain, Weekend Morning": {
        "AQI": 60, "PM2_5": 25, "PM10": 40, "temperature": 25, "humidity": 95,
        "wind_speed": 25, "pressure": 998, "rainfall_rate_mm_per_h": 25.0,
        "calendar": {"hour": 9, "day_of_week": 6, "month": 11, "is_weekend": 1},
    },
}
# Traffic's own feature set uses raw rainfall_mm (this hour) and rainfall_6h_sum, and calendar keys
# hour/day_of_week/is_weekend — reuse the same scenario dict, mapped on the fly in Section 6.
list(SCENARIOS.keys())


## 6. Run the engine across all zones × scenarios

In [ ]:
def predict_aqi_risk(zone_row, scenario, horizon):
    row = build_feature_row(aqi_metadata["feature_columns"], scenario, zone_row)
    X = pd.DataFrame([row])[aqi_metadata["feature_columns"]]
    pred_aqi = float(aqi_models[horizon].predict(X)[0])
    return pred_aqi, aqi_to_risk(pred_aqi)

def predict_flood_risk(zone_row, scenario, horizon):
    algo = flood_best_algo[horizon]
    cols = flood_metadata["feature_columns"]
    row = build_feature_row(cols, scenario, zone_row)
    model = flood_models[horizon][algo]
    if algo == "rf":
        df = pd.DataFrame([row])
        df["zone_id"] = zone_row["zone_id"]  # RF wants the string, one-hot encoded below
        X = pd.get_dummies(df, columns=["zone_id"]).reindex(columns=model.feature_names_in_, fill_value=0)
    elif algo == "logreg":
        lr_cols = [c for c in cols if c != "zone_id"]
        X = pd.DataFrame([row])[lr_cols]
    else:  # xgboost
        X = pd.DataFrame([row])[cols]
    prob = float(model.predict_proba(X)[0, 1])
    return algo, prob, prob_to_risk(prob)

def predict_traffic_risk(zone_row, scenario, horizon):
    algo = traffic_best_algo[horizon]
    cols = traffic_metadata["feature_columns"]
    traffic_scenario = {
        "rainfall_mm": scenario["rainfall_rate_mm_per_h"],
        "rainfall_6h_sum": scenario["rainfall_rate_mm_per_h"] * 6,
        "calendar": {
            "hour": scenario["calendar"]["hour"],
            "day_of_week": scenario["calendar"]["day_of_week"],
            "is_weekend": scenario["calendar"]["is_weekend"],
        },
    }
    row = build_feature_row(cols, traffic_scenario, zone_row)
    model = traffic_models[horizon][algo]
    if algo == "rf":
        df = pd.DataFrame([row])
        df["zone_id"] = zone_row["zone_id"]
        X = pd.get_dummies(df, columns=["zone_id"]).reindex(columns=model.feature_names_in_, fill_value=0)
    elif algo == "logreg":
        lr_cols = [c for c in cols if c != "zone_id"]
        X = pd.DataFrame([row])[lr_cols]
    else:
        X = pd.DataFrame([row])[cols]
    prob = float(model.predict_proba(X)[0, 1])
    return algo, prob, prob_to_risk(prob)

results = []
for scenario_name, scenario in SCENARIOS.items():
    for _, zone_row in zone_static_df.iterrows():
        pred_aqi, aqi_risk = predict_aqi_risk(zone_row, scenario, horizon=6)
        flood_algo, flood_prob, flood_risk = predict_flood_risk(zone_row, scenario, horizon=6)
        traffic_algo, traffic_prob, traffic_risk = predict_traffic_risk(zone_row, scenario, horizon=1)
        total, category = combine_urban_risk(aqi_risk, flood_risk, traffic_risk)
        results.append({
            "scenario": scenario_name, "zone_name": zone_row["zone_name"],
            "predicted_aqi_6h": round(pred_aqi, 1), "aqi_risk": round(aqi_risk, 1),
            "flood_model": flood_algo, "flood_prob_6h": round(flood_prob, 4), "flood_risk": round(flood_risk, 1),
            "traffic_model": traffic_algo, "traffic_prob_1h": round(traffic_prob, 4), "traffic_risk": round(traffic_risk, 1),
            "urban_risk_score": round(total, 1), "risk_category": category,
        })

risk_df = pd.DataFrame(results)
risk_df.sort_values(["scenario", "urban_risk_score"], ascending=[True, False]).head(20)


## 7. Anomaly detection — Isolation Forest
**Demo-fit only.** With just 3 scenarios × 20 zones (60 rows) there's no real historical distribution to learn from — this section proves the mechanism and interface work; M6 should refit `IsolationForest` on real historical hourly data before this is trusted for anything.

In [ ]:
from sklearn.ensemble import IsolationForest

anomaly_feature_cols = [
    "elevation_m", "water_body_distance_km", "historical_inundation_count",
    "road_density_score", "distance_to_congestion_hotspot_km",
]
scenario_feature_cols = ["AQI", "PM2_5", "PM10", "temperature", "humidity", "wind_speed", "pressure", "rainfall_rate_mm_per_h"]

anomaly_rows = []
for scenario_name, scenario in SCENARIOS.items():
    for _, zone_row in zone_static_df.iterrows():
        r = {c: zone_row[c] for c in anomaly_feature_cols}
        r.update({c: scenario[c] for c in scenario_feature_cols})
        r["scenario"] = scenario_name
        r["zone_name"] = zone_row["zone_name"]
        anomaly_rows.append(r)

anomaly_df = pd.DataFrame(anomaly_rows)
X_anomaly = anomaly_df[anomaly_feature_cols + scenario_feature_cols].fillna(0)

iso_forest = IsolationForest(n_estimators=200, contamination=0.1, random_state=42)
iso_forest.fit(X_anomaly)

anomaly_df["anomaly_score_raw"] = -iso_forest.score_samples(X_anomaly)  # higher = more unusual
lo, hi = anomaly_df["anomaly_score_raw"].min(), anomaly_df["anomaly_score_raw"].max()
anomaly_df["anomaly_severity_0_100"] = ((anomaly_df["anomaly_score_raw"] - lo) / max(hi - lo, 1e-9) * 100).round(1)
anomaly_df["is_anomaly"] = iso_forest.predict(X_anomaly) == -1

# "Expected vs observed": z-score of each feature relative to THIS demo set's own mean/std —
# illustrative only, given the tiny sample; a real baseline needs real historical hours (M6).
z_scores = (X_anomaly - X_anomaly.mean()) / X_anomaly.std(ddof=0).replace(0, 1)

top_anomalies = anomaly_df.sort_values("anomaly_severity_0_100", ascending=False).head(5)
for _, row in top_anomalies.iterrows():
    idx = row.name
    top_features = z_scores.loc[idx].abs().sort_values(ascending=False).head(3)
    print(f"{row['zone_name']} — {row['scenario']} | severity={row['anomaly_severity_0_100']}")
    for feat, z in top_features.items():
        print(f"    {feat}: observed={X_anomaly.loc[idx, feat]:.2f}  (z={z:+.2f} vs this demo set's mean)")
    print()

top_anomalies[["zone_name", "scenario", "anomaly_severity_0_100", "is_anomaly"]]


## 8. Save the risk engine

In [ ]:
import os
os.makedirs("models", exist_ok=True)

joblib.dump(iso_forest, "models/anomaly_isolation_forest_DEMO_FIT.joblib")
risk_df.to_csv("models/risk_engine_demo_scenarios.csv", index=False)

engine_config = {
    "scope_note": (
        "M4 combines M1/M2/M3 model outputs into a 0-100 Urban Risk Score and demonstrates an anomaly-"
        "detection layer. It does NOT ingest live data (that's M6) — Section 6's results are illustrative "
        "scenarios, not real predictions for a real moment in time."
    ),
    "risk_weights": RISK_WEIGHTS,
    "risk_weights_rationale": (
        "Documented judgment call, not an official standard. Flood weighted highest (acute safety risk, "
        "backed by real GCC inundation history in M2). AQI next (chronic health burden). Traffic lowest "
        "(rarely acutely dangerous)."
    ),
    "risk_categories": RISK_CATEGORIES,
    "aqi_rescale": "risk = min(100, predicted_AQI / 5) — linear rescale of the existing 0-500 index",
    "flood_model_selected_per_horizon": flood_best_algo,
    "traffic_model_selected_per_horizon": traffic_best_algo,
    "model_selection_method": "argmax test ROC-AUC per horizon, read dynamically from each milestone's own saved metadata",
    "zone_id_encoding_assumption": (
        "AQI and flood/traffic XGBoost models assumed trained on zone_id.astype('category').cat.codes "
        "(pandas default = alphabetically sorted unique values). Verify against actual M1 training code "
        "if a specific zone's AQI/flood predictions look inconsistent."
    ),
    "anomaly_detector_status": "DEMO-FIT on 60 illustrative rows only — refit on real historical hourly data in M6 before production use",
}

with open("models/risk_engine_metadata.json", "w") as f:
    json.dump(engine_config, f, indent=2, default=str)

print("Saved risk engine config, demo results, and demo-fit anomaly model to ./models/")
print(json.dumps(engine_config["scope_note"], indent=2))
